In [14]:
import pandas as pd

data = pd.read_csv("C:/Users/Baby Alekya/OneDrive/Documents/fraud_review_dataset.csv")

data.head()

,customer_id,review_id,product_id,star_rating,helpful_votes,total_votes,verified_purchase,review_body,review_date
0,32158956,R1KKOXHNI8MSXU,B01KL6O72Y,4,0,0,Y,"These Really Do Work Great, But You Do Need To...",14-01-2013
1,2714559,R26SP2OPDK4HT7,B01ID3ZS5W,5,1,2,Y,I love this dress. Absolute favorite for winte...,04-03-2014
2,12608825,RWQEDYAX373I1,B01I497BGY,5,0,0,Y,"Nice socks, great colors, just enough support ...",12-07-2015
3,25482800,R231YI7R4GPF6J,B01HDXFZK6,5,0,0,Y,"I bought this for my husband and WOW, this is ...",03-06-2015
4,9310286,R3KO3W45DD0L1K,B01G6MBEBY,5,0,0,Y,Perfect dress and the customer service was awe...,12-06-2015


In [15]:
data.columns

Index(['customer_id', 'review_id', 'product_id', 'star_rating',
       'helpful_votes', 'total_votes', 'verified_purchase', 'review_body',
       'review_date'],
      dtype='object')

In [16]:
#Remove unwanted columns 

data = data[['customer_id','product_id','star_rating',
             'helpful_votes','total_votes','verified_purchase',
             'review_body','review_date']]

In [17]:
# Creation of helpful_vote ratio

data['helpful_ratio'] = data['helpful_votes'] / (data['total_votes'] + 1)

In [18]:
#Conversion of date column to years and month

data['review_date'] = pd.to_datetime(data['review_date'], format='%d-%m-%Y')

In [19]:
data['review_year'] = data['review_date'].dt.year
data['review_month'] = data['review_date'].dt.month

In [20]:
#Convert verified_purchase

data['verified_purchase'] = data['verified_purchase'].map({'Y':1, 'N':0})

In [21]:
#Detect extreme ratings

data['extreme_rating'] = data['star_rating'].apply(lambda x: 1 if x==1 or x==5 else 0)

In [22]:
#Detect duplicate_reviews

data['duplicate_review'] = data.duplicated(subset=['review_body']).astype(int)

In [23]:
data = data.dropna(subset=['review_body'])

data['review_length'] = data['review_body'].astype(str).apply(len)

data['short_review'] = data['review_length'].apply(lambda x: 1 if x < 20 else 0)

In [24]:
from textblob import TextBlob
import numpy as np

# Sentiment Analysis
data['sentiment_score'] = data['review_body'].apply(
    lambda x: TextBlob(x).sentiment.polarity
)
# Sentiment Label 
data['sentiment_label'] = np.where(
    data['sentiment_score'] > 0, "Positive",
    np.where(data['sentiment_score'] < 0, "Negative", "Neutral")
)

# Rating Sentiment Mismatch

data['rating_sentiment_mismatch'] = (
    ((data['star_rating'] >= 4) & (data['sentiment_score'] < 0)) |
    ((data['star_rating'] <= 2) & (data['sentiment_score'] > 0))
).astype(int)

# Fraud Score Calculation
data['fraud_score'] = (
    (1 - data['verified_purchase']) +
    data['duplicate_review'] +
    data['extreme_rating'] +
    data['short_review'] +
    (data['helpful_ratio'] < 0.2).astype(int) +
    data['rating_sentiment_mismatch']
)

# Fraud Label
data['fraud_label'] = np.where(
    data['fraud_score'] >= 3, "Fraud", "Genuine"
)
print(data[['review_body','star_rating','sentiment_label','fraud_score','fraud_label']].head())
data.to_csv("final_fraud_reviews_dataset.csv", index=False)

                                         review_body  star_rating  \
0  These Really Do Work Great, But You Do Need To...            4   
1  I love this dress. Absolute favorite for winte...            5   
2  Nice socks, great colors, just enough support ...            5   
3  I bought this for my husband and WOW, this is ...            5   
4  Perfect dress and the customer service was awe...            5   

  sentiment_label  fraud_score fraud_label  
0        Positive            1     Genuine  
1        Positive            1     Genuine  
2        Positive            2     Genuine  
3        Positive            2     Genuine  
4        Positive            2     Genuine  
